# Dask cluster smoke test

Synthetic data only (no dependency on a real Discogs ingest). Proves this
notebook is talking to the real `dask-scheduler`/`dask-worker` Swarm
services in `docker-stack.dask.yml`, not silently falling back to Dask's
local-threads scheduler inside the Jupyter container itself.


In [10]:
from dask.distributed import Client

client = Client("tcp://dask-scheduler:8786")
client

<Client: 'tcp://172.18.0.4:8786' processes=2 threads=4, memory=1.00 GiB>

In [11]:
# If this prints only the Jupyter container's own hostname, something is
# wrong -- it should show one or more dask-worker container hostnames.
print("Workers registered with the scheduler:")
for addr, info in client.scheduler_info()["workers"].items():
    print(f"  {addr}  host={info['host']}  nthreads={info['nthreads']}")

Workers registered with the scheduler:
  tcp://10.0.2.6:35439  host=10.0.2.6  nthreads=2
  tcp://10.0.2.7:45005  host=10.0.2.7  nthreads=2


In [12]:
import dask.array as da

x = da.random.random((4000, 4000), chunks=(1000, 1000))
result = (x + x.T).sum().compute()
print("Sum:", result)

Sum: 15997208.470856097


In [13]:
# Confirm partitions actually ran on worker processes, not this notebook's
# own kernel process.
import dask.dataframe as dd
import pandas as pd
from distributed import get_worker

pdf = pd.DataFrame({"n": range(200)})
ddf = dd.from_pandas(pdf, npartitions=8)
# meta= is required here: without it, Dask first calls this lambda locally
# (on the client, not a real worker) to infer the output schema, and
# get_worker() only works inside an actual worker process -- that inference
# dry-run fails with "No worker found" before any real distributed
# execution happens. Confirmed live.
worker_per_partition = ddf.map_partitions(
    lambda part: pd.Series([get_worker().address] * len(part)),
    meta=("worker_address", "object"),
).compute()
print("Distinct workers that processed a partition:")
print(sorted(worker_per_partition.unique()))

Distinct workers that processed a partition:
['tcp://10.0.2.6:35439', 'tcp://10.0.2.7:45005']
